# EV Battery SOH — Deep-learning experiments (Google Colab)

Use this notebook for LSTM, 1D-CNN and Transformer experiments. Your Windows PC does **not** need a GPU. Run data inspection, preprocessing and classical ML locally; run deep-learning experiments here on a Colab GPU.

## Before running
1. Upload the extracted `ev-battery-soh` project folder to `MyDrive/ev-battery-soh`.
2. Upload the prepared `data/processed/unified.csv` and any required source data to the same project folder.
3. In Colab select **Runtime → Change runtime type → T4 GPU** (if available).
4. Run cells in order. Results are saved in the project folder on Google Drive.

Do not upload this project's ZIP if you have already extracted it; Colab needs the folder containing `pyproject.toml`.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/ev-battery-soh')
assert (PROJECT_DIR / 'pyproject.toml').exists(), f'Project folder not found: {PROJECT_DIR}'
%cd /content/drive/MyDrive/ev-battery-soh
print('Project:', PROJECT_DIR)

In [ ]:
# Install the project dependencies. Colab normally already has PyTorch installed.
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
import torch, pandas as pd
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU unavailable. Select Runtime > Change runtime type > T4 GPU, or set DEVICE=cpu for a small smoke test.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from battery_soh.data.schema import harmonize
DATA_PATH = PROJECT_DIR / 'data/processed/unified.csv'
assert DATA_PATH.exists(), f'Missing prepared data: {DATA_PATH}. Prepare the real datasets on your PC first.'
df = harmonize(pd.read_csv(DATA_PATH))
print('Rows:', len(df))
print('Datasets:', sorted(df.dataset.unique()))
print('Batteries by dataset:')
display(df.groupby('dataset').agg(batteries=('battery_id','nunique'), observations=('cycle','size'), min_soh=('soh','min'), max_soh=('soh','max')))
assert len(df.dataset.unique()) >= 2, 'Need at least two datasets for transfer experiments.'

## 1. Pilot run — one seed, cross-dataset transfer
Run this first. Check `failures.csv` and confirm the training/test battery counts before launching the larger study. The current common feature baseline is `cycle`; do not add a feature until it is genuinely present and comparable across all selected datasets.

In [ ]:
from battery_soh.experiments.engine import benchmark
from battery_soh.evaluation.plots import make_figures
pilot_out = PROJECT_DIR / 'results/colab_deep_cross_pilot'
benchmark(df, experiment='cross', models=['lstm','cnn','transformer'], seeds=[42],
          features=['cycle'], window=4,
          deep_config={'epochs': 20, 'patience': 5, 'device': DEVICE},
          output=pilot_out, strict=False)
make_figures(pilot_out)
display(pd.read_csv(pilot_out / 'metrics.csv'))
display(pd.read_csv(pilot_out / 'failures.csv'))

## 2. Full 5×5 matrix — deep models, five seeds
Run only after the pilot has no unexplained failures. This includes within-dataset diagonal tests and cross-dataset off-diagonal tests. It can take a while even on a GPU.

In [ ]:
assert DEVICE == 'cuda', 'Select Runtime > Change runtime type > T4 GPU before the full deep-learning benchmark.'
full_matrix_out = PROJECT_DIR / 'results/colab_deep_matrix_5seeds'
benchmark(df, experiment='matrix', models=['lstm','cnn','transformer'],
          seeds=[11,22,33,44,55], features=['cycle'], window=4,
          deep_config={'epochs': 30, 'patience': 5, 'device': DEVICE},
          output=full_matrix_out, strict=False)
make_figures(full_matrix_out)
print('Saved:', full_matrix_out)
display(pd.read_csv(full_matrix_out / 'failures.csv').head(30))

## 3. Leave-one-dataset-out deep-learning tests
Each target dataset is excluded from source training. This is a separate experiment from the 5×5 pairwise matrix.

In [ ]:
assert DEVICE == 'cuda', 'Select a Colab GPU runtime before the full LODO deep-learning benchmark.'
lodo_out = PROJECT_DIR / 'results/colab_deep_lodo_5seeds'
benchmark(df, experiment='lodo', models=['lstm','cnn','transformer'],
          seeds=[11,22,33,44,55], features=['cycle'], window=4,
          deep_config={'epochs': 30, 'patience': 5, 'device': DEVICE},
          output=lodo_out, strict=False)
make_figures(lodo_out)
display(pd.read_csv(lodo_out / 'metrics.csv').head(30))
display(pd.read_csv(lodo_out / 'failures.csv').head(30))

## Validation checklist
- Confirm no source/target battery ID overlap.
- Confirm target dataset was never used for model fitting in cross/LODO tests.
- Review every row in `failures.csv`.
- Compare MAE/RMSE/MAPE with the CPU classical baselines.
- Do not report the pilot as the final five-seed result.
- Save the notebook output and all CSVs in Drive.